In [1]:
import torch
import full_structure_snap as fss
import os
from pymatgen.core import Structure
import torch


wavelength = 1.54184
q_max = 5.76722732255
q_min = 0.355794747244
num_steps = 850
results_directory = "./test_results/"
structure_directory = "./test_cifs/"

gt_cif_list = []
for cif in os.listdir(structure_directory):
    if cif.split("_")[0] == "gt":
        gt_cif_list.append(cif)

plot_dictionary = {'plot_progress': True,
                        'plot_freq': 1000,
                        'graph_losses': True}
SS = fss.FullStructureSnapper(q_max= q_max, q_min = q_min, wavelength= wavelength, plot_dictionary = plot_dictionary)

batch_size = 16
bin_batch_size = 8
UVW = torch.tensor([[0,0,0.01]]).cuda()

cif_batch = gt_cif_list[0:2*batch_size]
print(cif_batch)
gt_batches = []
for cif in cif_batch:
    gt_structure = Structure.from_file(structure_directory + f"{cif}")
    gt_batch = SS.structure_to_batch(gt_structure)
    gt_batches.append(gt_batch)
batch = SS.merge_batch(gt_batches)

gt_patterns = SS.batch_split_bin_pattern_theta(SS.batch_split_diffraction_calc(batch, batch_size), bin_batch_size, UVW = UVW, num_steps = num_steps)

pred_batches = []
for cif in cif_batch:
    if len(cif.split("'")) > 1:
        cif = cif.split("'")[1]
        cif = cif[2:]
        cif = "'pred" + cif + "'"
    else:
        cif = cif[2:]
        cif = "pred" + cif
    pred_structure = Structure.from_file(structure_directory + cif)
    pred_batch = SS.structure_to_batch(pred_structure)
    pred_batches.append(pred_batch)
batch = SS.merge_batch(pred_batches)

['gt_BaAgBi_583.cif', 'gt_BaCuO2_91.cif', 'gt_Ba3WN3_640.cif', 'gt_LuMg3_368.cif', 'gt_Ba2PuInO6_490.cif', 'gt_Ti2ReOs_415.cif', 'gt_ScInPt2_357.cif', 'gt_LiCd2Cu_275.cif', 'gt_LaTiO3_200.cif', 'gt_CaNi5_301.cif', 'gt_NaTbTl2_35.cif', 'gt_Li2Ni3TeO8_18.cif', 'gt_SrLiGaF6_131.cif', 'gt_W2NCl8_615.cif', 'gt_Mg2SiO4_244.cif', 'gt_VGaOs2_136.cif', 'gt_YbBRh3_363.cif', 'gt_Ag2BiSbTe4_102.cif', 'gt_ScAgSe2_121.cif', 'gt_Mg2ZnO3_376.cif', 'gt_CsLuCdTe3_361.cif', 'gt_YbGaPd2_34.cif', 'gt_Ba(PIr)2_310.cif', 'gt_SnTe2Pb_440.cif', 'gt_Li2CoSnO4_104.cif', 'gt_Ag2Se_590.cif', 'gt_Ca(BIr)2_387.cif', 'gt_Rb2TlAsBr6_165.cif', 'gt_Ca(AsO3)2_525.cif', 'gt_V2FeS4_458.cif', 'gt_Sc7NCl12_665.cif', 'gt_LiPr2OsO6_370.cif']


/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


In [2]:
batch

{'lengths': tensor([[ 4.7894,  4.8293,  9.7834],
         [ 4.2065,  4.2575,  4.5808],
         [ 5.4261,  7.8402,  7.8122],
         [ 4.7442,  4.7301,  4.7839],
         [ 5.9202,  5.9748,  6.0762],
         [ 4.3466,  4.3103,  4.3138],
         [ 4.6253,  4.5785,  4.5687],
         [ 4.5792,  4.5569,  4.5492],
         [ 5.5715,  5.7420,  7.8629],
         [ 4.0259,  4.6200,  4.8607],
         [ 5.3478,  5.4107,  5.3857],
         [ 6.0612,  5.9542,  6.1299],
         [ 5.1218,  5.2132, 10.6190],
         [ 6.6694,  6.6940,  9.4811],
         [ 5.8958,  5.8465,  6.0688],
         [ 4.4786,  4.4845,  4.4271],
         [ 4.2838,  4.3426,  4.2030],
         [ 6.2061,  6.1670,  6.9854],
         [ 3.9043,  3.9000,  6.6454],
         [ 2.8207,  2.7009,  8.6087],
         [ 4.3438,  9.0886, 12.4913],
         [ 4.7529,  4.7381,  4.6211],
         [ 3.7762,  3.9227,  6.3460],
         [ 6.1981,  6.4047,  6.7091],
         [ 5.6157,  6.0741,  6.1917],
         [ 4.0976,  7.0249,  7.8029],
 

In [ ]:
snapped_batch, final_losses = SS.full_structure_snap(gt_patterns, batch)
print(final_losses)
print(snapped_batch['lengths'].shape)